<a href="https://colab.research.google.com/github/sicfy-ai/shared_notebooks/blob/main/rag_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from bs4 import BeautifulSoup as Bs
import requests
import re

# https://www.irs.gov/prior-year-forms-and-instructions?find=Publ%20907&items_per_page=200&order=prior_year_products_picklist_revision_date&sort=desc

In [ ]:
html_response = requests.get("https://www.irs.gov/publications/p547")

In [ ]:
soup = Bs(html_response.text, "html.parser")

In [ ]:
book_div = soup.find("div", class_="book")
wanted_classes = {"article", "part", "titlepage"}
target_divs = [
    div for div in book_div.find_all("div", recursive=False)
    if any(cls in wanted_classes for cls in div.get("class", []))
]
filtered_soup = Bs("<div class='filtered'></div>", "html.parser")
container = filtered_soup.div

for div in target_divs:
    container.append(div)

# Now you have just the target content
reduced_html = str(filtered_soup)

In [ ]:
soup_v1 = Bs(reduced_html, "html.parser")
decomposable_tags: set = {"script", "style", "noscript", "iframe", "svg", "canvas", "link", "meta", "ins", "object",
                          "embed"}
# Remove unnecessary elements
for tag in soup_v1.find_all(decomposable_tags):
    tag.decompose()

for input_tag in soup_v1.find_all("input", {"type": "hidden", "name": "javax.faces.ViewState"}):
    input_tag.decompose()

# Also remove by ID to be thorough
viewstate_elem = soup_v1.find(id="javax.faces.ViewState")
if viewstate_elem:
    viewstate_elem.decompose()

for noisy in soup_v1.select(
        "[class*='ad-'], [id*='ad-'], .share, .social, .gpt, .banner, .cookie"
):
    noisy.decompose()

for txt in soup_v1.find_all(string=True):
    if re.search(r"\s{3,}", txt):
        txt.replace_with(re.sub(r"\s{3,}", " ", txt))


In [ ]:
from bs4 import NavigableString

# Use one soup object
for tag in soup_v1.find_all(['b', 'strong']):
    tag.unwrap()

for a_tag in soup_v1.find_all('a'):
    if not a_tag.get('href'):
        a_tag.decompose()
        continue

    link_text = a_tag.get_text(strip=True)
    href = a_tag.get('href', '').strip()

    if href.lower().startswith('http'):
        formatted_text = f"{link_text or href} [{href}]"
    else:
        formatted_text = link_text

    if formatted_text.strip() and formatted_text.strip() != "[]":
        a_tag.insert_after(NavigableString(formatted_text.strip()))
    a_tag.decompose()

for p_tag in soup_v1.find_all("p"):
    if not p_tag.get_text(strip=True):
        p_tag.decompose()


In [ ]:
import re
import unicodedata

def clean_text(text):
    # Replace smart quotes with straight ones
    text = text.replace("’", "'").replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("−", '-')
    text = text.replace("×", 'x')
    text = text.replace("÷", '/')
    text = text.replace("®", '(R)')
    text = text.replace("⁄", '/')

    # Replace en dash and em dash with normal dash
    text = text.replace("–", "-").replace("—", "-")

    # Remove non-breaking space, zero-width space, soft hyphen
    text = text.replace("\xa0", " ").replace("\u200b", "").replace("\u00ad", "")

    # Optional: remove excess whitespace
    # text = re.sub(r"[ \t\r\f\v]+", " ", text)         # collapse all weird spaces
    # text = re.sub(r"\s*\n\s*", "\n", text)            # trim around newlines
    # text = text.strip()

    return text

# text = soup_v1.get_text()
text = unicodedata.normalize("NFKC", str(soup_v1))
text = unicodedata.normalize('NFKD', text)
cleaned_text = clean_text(text)
ascii_safe_text = cleaned_text.encode("ascii", errors="ignore").decode("ascii")


In [ ]:
soup_3 = Bs(ascii_safe_text, "html.parser")
content_divs = soup_3.find_all(["div", "article"], class_=["article", "content", "main-content"])
# print(len(content_divs))
sections = {}
heading_elements_list = []
for content_div in content_divs:
  if content_div:
      heading_elements = content_div.find_all(
          ["h2", "h3", "h4"], class_=["title", "role-hd1", "section-title"]
      )
      heading_elements_list.extend(heading_elements)
heading_elements_list

[<h2 class="title role-highlight" style="clear: both">Future Developments</h2>,
 <h2 class="title role-highlight" style="clear: both">What's New</h2>,
 <h2 class="title role-highlight" style="clear: both">Reminders</h2>,
 <h4 class="title role-intro">Introduction</h4>,
 <h4 class="title role-hd1">Casualty</h4>,
 <h3 class="title"><span class="inlinep">Note.</span></h3>,
 <h4 class="title role-hd2">Special Procedure for Damage From Corrosive Drywall</h4>,
 <h3 class="title">Note.</h3>,
 <h4 class="title role-hd1">Theft</h4>,
 <h4 class="title role-hd1">Loss on Deposits</h4>,
 <h4 class="title role-hd1">Proof of Loss</h4>,
 <h4 class="title role-hd1">Figuring a Loss</h4>,
 <h4 class="title role-hd2">Decrease in FMV</h4>,
 <h4 class="title role-hd3">Figuring Decrease in FMV-Items To Consider</h4>,
 <h4 class="title role-hd3">Special Procedure-Safe Harbor Methods for Determining Casualty and Theft Losses</h4>,
 <h4 class="title role-hd3">Figuring Decrease in FMV-Items Not To Consider</h4>,

In [ ]:
len(heading_elements_list)

46

In [ ]:
title_candidates = [
        soup_3.find("title"),
        soup_3.find("h1"),
        soup_3.find("meta", attrs={"name": "twitter:title"}),
    ]
title_candidates
doc_structure = {"subtitle": "Tax Information Document", "title": "IRS Publication"}

for candidate in title_candidates:
    if candidate:
        if getattr(candidate, "content", None):
            doc_structure["title"] = candidate["content"]
        elif candidate.text:
            doc_structure["title"] = candidate.text.strip()

        if doc_structure["title"]:
            break

subtitle_elem = soup.find("meta", attrs={"name": "dc.title"})
if subtitle_elem and subtitle_elem.get("content"):
    doc_structure["subtitle"] = subtitle_elem["content"]


In [ ]:
def get_sections(heading_elements_list, sections):
    section_id = 1
    for heading in heading_elements_list:
        section_title = heading.text.strip()
        if section_title:
            # Find the parent that contains both the heading and following content
            parent_element = heading

            for _ in range(3):  # Try going up to 3 levels up to find common parent
                if parent_element.parent:
                    parent_element = parent_element.parent

            # Get all content following this heading until the next heading
            content = []
            next_element = heading

            # First, collect any direct content within the heading's container
            while next_element.next_sibling:
                next_element = next_element.next_sibling
                if next_element.name in ["h2", "h3", "h4"] and any(
                    cls in next_element.get("class", [])
                    for cls in ["title", "role-hd1", "section-title"]
                ):
                    break
                if next_element.name:
                    content.append(str(next_element))

            # If no content was found within the container, look for content after the container
            if not content and parent_element.next_sibling:
                next_element = parent_element
                while next_element.next_sibling:
                    next_element = next_element.next_sibling
                    if isinstance(next_element, str) and not next_element.strip():
                        continue  # Skip empty text nodes
                    if next_element.name in ["h2", "h3", "h4"] and any(
                        cls in next_element.get("class", [])
                        for cls in ["title", "role-hd1", "section-title"]
                    ):
                        break
                    if next_element.name:
                        content.append(str(next_element))

            # Create section structure
            section_info = {
                "id": str(section_id),
                "title": section_title,
                "content": "".join(content).replace("  ", "").replace("\n", ""),
                "subsections": {},
            }

            sections[str(section_id)] = section_info
            section_id += 1

get_sections(heading_elements_list, sections)

In [ ]:
doc_structure["sections"] = sections
num_sections = len(doc_structure.get("sections", {}))
num_sections
sections

n = 0
for i, section in sections.items():
  n += len(section["content"])

n

508198

In [ ]:
sections

{'1': {'id': '1',
  'title': 'Future Developments',
  'content': '<p>For the latest information about developments related to Pub. 547, such as legislation enacted after it was published, go to IRS.gov/Pub547 [https://www.irs.gov/forms-pubs/about-publication-547].</p>',
  'subsections': {}},
 '2': {'id': '2',
  'title': "What's New",
  'content': '<p class="item">Extended disaster relief benefits. The Federal Disaster Tax Relief Act of 2023 extended the special rules and return procedures for personal casualty losses attributable to certain major federal disasters declared between February 26, 2021, and February 10, 2025. Qualified disaster losses can be claimed on Form 4684, Casualties and Thefts. For more information, see Qualified disaster loss, later.</p><p class="item">Qualified wildfire relief payments. Certain relief payments received between 2020 and 2025 following a wildfire disaster are not taxable. For more information, see Qualified wildfire relief payments, later.</p><p cl

Older Version

In [ ]:
# soup = Bs(content_div, "html.parser")
soup = content_div
decomposable_tags: set = {"script", "style", "noscript", "iframe", "svg", "canvas", "link", "meta", "ins", "object", "embed"}
# Remove unnecessary elements
for tag in soup.find_all(decomposable_tags):
    tag.decompose()

for input_tag in soup.find_all("input", {"type": "hidden", "name": "javax.faces.ViewState"}):
    input_tag.decompose()

# Also remove by ID to be thorough
viewstate_elem = soup.find(id="javax.faces.ViewState")
if viewstate_elem:
    viewstate_elem.decompose()

for noisy in soup.select(
        "[class*='ad-'], [id*='ad-'], .share, .social, .gpt, .banner, .cookie"
):
    noisy.decompose()

for txt in soup.find_all(string=True):
    if re.search(r"\s{3,}", txt):
        txt.replace_with(re.sub(r"\s{3,}", " ", txt))

inline_tags = ['em', 'strong', 'span', 'b']
for tag in soup.find_all(inline_tags):
    tag.replace_with(tag.get_text())

for a_tag in soup.find_all('a'):
    # Extract the text and href
    link_text = a_tag.get_text(strip=True)
    href = a_tag.get('href', '')

    # Replace the <a> tag with the desired format
    if href.startswith('http'):
      formatted_text = f"{link_text} [{href}]"
      if formatted_text != []:
        print("href a8", formatted_text)
        a_tag.replace_with(formatted_text)

    # Remove <a> tags that don't have a string or href
    if not a_tag.string and not a_tag.get('href'):
      a_tag.decompose()

print(soup)

In [ ]:
soup

<div class="content">
<div class="field field--name-body field--type-text-with-summary field--label-hidden field--item"><div class="row">
<div class="col-md-4">
<ul>
<li>Publication 17 - Introductory Material [#idm140408606312624]
<ul class="collapse in" id="idm140408606312624">
<li>What's New [#en_US_2024_publink1000170255]</li>
<li>Reminders [#en_US_2024_publink1000290775]</li>
<li>Introduction [#idm140408605389632]
<ul class="collapse in" id="">
<li>How this publication is arranged. [#en_US_2024_publink1000170345]</li>
<li>What is in this publication. [#en_US_2024_publink1000170346]</li>
<li>Icons. [#en_US_2024_publink1000170347]</li>
<li>What is not covered in this publication. [#en_US_2024_publink1000170348]</li>
<li>Help from the IRS. [#en_US_2024_publink1000170349]</li>
<li>Comments and suggestions. [#en_US_2024_publink1000139502]</li>
<li>Getting answers to your tax questions. [#en_US_2024_publink1000139503]</li>
<li>Getting tax forms, instructions, and publications. [#en_US_20

In [ ]:
sections = {}

# Look for various heading patterns used in publications
heading_elements = []
if content_div:
    heading_elements = content_div.find_all(
        ["h2", "h3", "h4"], class_=["title", "role-hd1", "section-title"]
    )

# If no headings found with specific classes, try all headings
if not heading_elements:
    heading_elements = soup.find_all(["h2", "h3", "h4"])


def get_sections(heading_elements, sections):
    section_id = 1
    for heading in heading_elements:
        section_title = heading.text.strip()
        if section_title:
            # Find the parent that contains both the heading and following content
            parent_element = heading

            for _ in range(3):  # Try going up to 3 levels up to find common parent
                if parent_element.parent:
                    parent_element = parent_element.parent

            # Get all content following this heading until the next heading
            content = []
            next_element = heading

            # First, collect any direct content within the heading's container
            while next_element.next_sibling:
                next_element = next_element.next_sibling
                if next_element.name in ["h2", "h3", "h4"] and any(
                    cls in next_element.get("class", [])
                    for cls in ["title", "role-hd1", "section-title"]
                ):
                    break
                if next_element.name:
                    content.append(str(next_element))

            # If no content was found within the container, look for content after the container
            if not content and parent_element.next_sibling:
                next_element = parent_element
                while next_element.next_sibling:
                    next_element = next_element.next_sibling
                    if isinstance(next_element, str) and not next_element.strip():
                        continue  # Skip empty text nodes
                    if next_element.name in ["h2", "h3", "h4"] and any(
                        cls in next_element.get("class", [])
                        for cls in ["title", "role-hd1", "section-title"]
                    ):
                        break
                    if next_element.name:
                        content.append(str(next_element))

            # Create section structure
            section_info = {
                "id": str(section_id),
                "title": section_title,
                "content": "".join(content).replace("  ", "").replace("\n", ""),
                "subsections": {},
            }

            sections[str(section_id)] = section_info
            section_id += 1

get_sections(heading_elements, sections)



In [ ]:
print(sections)
# print(heading_elements)
len(sections)

{'1': {'id': '1', 'title': "[]What's New", 'content': '<p>This section summarizes important tax changes that took effect in 2024. Most of these changes are discussed in more detail throughout this publication.</p><p class="item"> []Future developments. For the latest information about the tax law topics covered in this publication, such as legislation enacted after it was published, go to IRS.gov/Pub17 [https://www.irs.gov/pub17].</p><p class="item"> []Who must file. Generally, the amount of income you can receive before you must file a return has been increased. For more information, see chapter 1 [#en_US_2024_publink1000170357], later.</p><p class="item"> []Due date of return. File Form 1040 or 1040-SR by April 15, 2025. See chapter 1 [#en_US_2024_publink1000170357], later.</p><p class="item"> []Additonal child tax credit (ACTC) amount increased. The maximum ACTC amount has increased to $1,700 for each qualifying child.</p><p class="item"> []Standard deduction amount increased. For 2

439

In [ ]:
content_div = soup.find(['div', 'article'], class_=['content', 'main-content'])
str(content_div).replace("\n", "")
heading_elements = content_div.find_all(['h2', 'h3', 'h4'],
                                                class_=['title', 'role-hd1', 'section-title'])
heading_elements
heading_elements[3].parent
heading_elements[3].next_sibling

'\n'

In [ ]:
sections = {}
section_id = 1
# Process each heading as a section
for heading in heading_elements:
    section_title = heading.text.strip()
    if section_title:
        # Find the parent that contains both the heading and following content
        parent_element = heading
        # TODO: getting difficult from here
        for _ in range(3):  # Try going up to 3 levels up to find common parent
            if parent_element.parent:
                parent_element = parent_element.parent

        # Get all content following this heading until the next heading
        content = []
        next_element = heading

        # First, collect any direct content within the heading's container
        while next_element.next_sibling:
            next_element = next_element.next_sibling
            if next_element.name in ['h2', 'h3', 'h4'] and any(cls in next_element.get('class', [])
                                                                for cls in ['title', 'role-hd1', 'section-title']):
                break
            if next_element.name:
                content.append(str(next_element))

        # If no content was found within the container, look for content after the container
        if not content and parent_element.next_sibling:
            next_element = parent_element
            while next_element.next_sibling:
                next_element = next_element.next_sibling
                if isinstance(next_element, str) and not next_element.strip():
                    continue  # Skip empty text nodes
                if next_element.name in ['h2', 'h3', 'h4'] and any(cls in next_element.get('class', [])
                                                                    for cls in
                                                                    ['title', 'role-hd1', 'section-title']):
                    break
                if next_element.name:
                    content.append(str(next_element))

        # Create section structure
        section_info = {
            'id': str(section_id),
            'title': section_title,
            'content': ''.join(content),
            'subsections': {}
        }

        sections[str(section_id)] = section_info
        section_id += 1

# If no sections found, create a single section from all content
if not sections:
    # logger.debug("  - No sections found, creating single section from document body")

    body = soup.find('body')
    if body:
        sections['1'] = {
            'id': '1',
            'title': doc_structure.get('title', 'Document Content'),
            'content': str(body),
            'subsections': {}
        }

doc_structure['sections'] = sections

In [ ]:
doc_structure

{'subtitle': 'Tax Information Document',
 'title': 'Publication 907 (2024), Tax Highlights for Persons With Disabilities | Internal Revenue Service',
 'sections': {'1': {'id': '1',
   'title': 'Future Developments',
   'content': '<p>For the latest information about developments related to Pub. 907, such as legislation enacted after this publication was published, go to IRS.gov/Pub907 (https://www.irs.gov/pub907).</p>',
   'subsections': {}},
  '2': {'id': '2',
   'title': 'What’s New',
   'content': '<p class="item">Annual contribution limit. For 2024, the maximum amount that can be contributed to your ABLE account is $18,000. Certain employed ABLE account beneficiaries may contribute a limited additional amount. See Contribution limitation, later.</p><p class="item">Retirement savings contributions credit (saver’s credit) income limits increased. For 2024, your modified adjusted gross income must be not more than $38,250 ($76,500 if married filing jointly; $57,375 if head of househol

In [ ]:
for heading in heading_elements:
    section_title = heading.text.strip()
    break

section_title

'Future Developments'

In [ ]:
html_response.text

soup = Bs(html_response.text, 'html.parser')
soup.prettify()

'<!DOCTYPE html>\n<html dir="ltr" lang="en" prefix="content: http://purl.org/rss/1.0/modules/content/  dc: http://purl.org/dc/terms/  foaf: http://xmlns.com/foaf/0.1/  og: http://ogp.me/ns#  rdfs: http://www.w3.org/2000/01/rdf-schema#  schema: http://schema.org/  sioc: http://rdfs.org/sioc/ns#  sioct: http://rdfs.org/sioc/types#  skos: http://www.w3.org/2004/02/skos/core#  xsd: http://www.w3.org/2001/XMLSchema# ">\n <head>\n  <meta charset="utf-8"/>\n  <link href="https://www.irs.gov/publications/p907" rel="canonical"/>\n  <meta content="index, follow" name="robots"/>\n  <meta content="United States Internal Revenue Services" name="rights"/>\n  <meta content="https://www.irs.gov/pub/image/logo_small.jpg" property="og:image:url"/>\n  <meta content="image/jpeg" property="og:image:type"/>\n  <meta content="IRS logo" property="og:image:alt"/>\n  <meta content="summary" name="twitter:card"/>\n  <meta content="" name="twitter:description"/>\n  <meta content="Publication 907 (2024), Tax Highl

Remove Script tags


In [ ]:
# Config
tags_to_remove = ['footer', 'link', 'script', 'meta', 'noscript', 'img']
tags_to_flatten = ["p", "td", "span", "a", "dt"]
tags_to_unwrap = ["blockquote", "em", "span"]
non_content_tags = ['hr', 'br']

# Process each "book" div
target_soup = soup.find_all("div", class_="book")  # or soup.select("div.book")

for book_soup in target_soup:
    # 1. Process <a> tags first
    for a_tag in book_soup.find_all('a'):
        href = a_tag.get('href', '')
        display_text = a_tag.get_text(strip=True)

        if href.startswith('https://'):
            a_tag.replace_with(f'{display_text} ({href})')
        else:
            a_tag.unwrap()

    # 2. Remove unwanted tags
    for tag in tags_to_remove:
        for match in book_soup.find_all(tag):
            match.decompose()

    # 3. Flatten tags like <p>, <td>, <span>
    for tag in tags_to_flatten:
        for match in book_soup.find_all(tag):
            # simplified_paragraph = ' '.join(match.get_text().split())
            # match.string.replace_with(simplified_paragraph)  # Fixes the problem!
            simplified_paragraph = ' '.join(match.get_text().split())

            # Remove all existing contents, then insert cleaned text
            match.clear()  # Clears the tag's content
            match.append(simplified_paragraph)  # Adds cleaned text


    # 4. Unwrap specified tags (e.g., blockquote, em)
    for tag in tags_to_unwrap:
        for match in book_soup.find_all(tag):
            match.unwrap()

    # 5. Remove empty divs after processing
    for empty_book in book_soup.find_all('div'):
        # Remove non-content tags first
        for tag in non_content_tags:
            for match in empty_book.find_all(tag):
                match.decompose()

        # Remove div if empty
        if not empty_book.get_text(strip=True):
            empty_book.decompose()

# Print updated divs
for div in target_soup:
    print(div.prettify())


<div class="book" lang="en">
 <div class="titlepage">
  <div>
   <h1>
    Publication 907 (2024), Tax Highlights for Persons With Disabilities
   </h1>
   <div>
    <p class="pubdate">
     For use in preparing 2024 Returns
    </p>
   </div>
  </div>
 </div>
 <div class="article">
  <div class="titlepage">
   <div>
    <div>
     <h1 class="title">
      Publication 907 - Introductory Material
     </h1>
    </div>
   </div>
  </div>
  <div class="section" id="idm139999555254800">
   <div class="titlepage">
    <div>
     <div>
      <h2 class="title role-highlight" style="clear: both">
       Future Developments
      </h2>
     </div>
    </div>
   </div>
   <p>
    For the latest information about developments related to Pub. 907, such as legislation enacted after this publication was published, go to IRS.gov/Pub907 (https://www.irs.gov/pub907).
   </p>
  </div>
  <div class="section" id="idm139999555252144">
   <div class="titlepage">
    <div>
     <div>
      <h2 class="title ro

now the document is processed and ready for advance extraction from selected html.

Create a display that takes processed document in smart way (display its tags, class, hover other attributes)

table can be removed, ignore, flatten
ul, ol can be removed, ignored, flatten


In [ ]:
review =